In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmartdata202611")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/movie_cast.json"

In [0]:
constructors_schema = StructType(fields=[
    StructField("movieId", IntegerType(), False),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True)
])


In [0]:
df_constructors = spark.read \
    .schema(constructors_schema) \
    .option("multiLine", True) \
    .json(ruta)

In [0]:
constructors_final_df = df_constructors.select(
    col("movieId").alias("movie_Id"),
    col("personId").alias("person_Id"),
    col("characterName"),
    col("genderId")
).withColumn("ingestion_date", current_timestamp())

In [0]:
constructors_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.movie_cast")